In [1]:
# ============================================================
# GRUPPO 1A — CARICAMENTO DATASET (Breast Cancer) + INFO + SPLIT
# ============================================================

# Import minimi per questa sezione
from sklearn.datasets import load_breast_cancer         # dataset di esempio
from sklearn.model_selection import train_test_split    # split train/test
import pandas as pd                                    # per value_counts() e stampa distribuzione

# 1) Carico il dataset (oggetto "Bunch" di sklearn)
data = load_breast_cancer()

# 2) Estraggo feature (X) e target (y)
X = data.data            # matrice: (n_campioni, n_feature) -> qui (569, 30)
y = data.target          # vettore etichette -> 0 = malignant, 1 = benign

# 3) Stampo info utili per capire i dati
print(f"Dimensione dataset: {X.shape}")                         # es: (569, 30)
print(f"Classi: {data.target_names}")                           # ['malignant' 'benign']
print(f"Distribuzione classi: {pd.Series(y).value_counts().to_dict()}")

# 4) Split train/test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% test
    random_state=42,      # riproducibilità
    stratify=y            # mantiene la proporzione delle classi
)

# 5) Controllo dimensioni split
print(f"Train: {X_train.shape[0]} campioni")
print(f"Test: {X_test.shape[0]} campioni")


Dimensione dataset: (569, 30)
Classi: ['malignant' 'benign']
Distribuzione classi: {1: 357, 0: 212}
Train: 455 campioni
Test: 114 campioni


In [2]:
# ============================================================
# GRUPPO 1B — CARICAMENTO DATASET (UCI Bank Marketing) + INFO + SPLIT
# ============================================================

# Import minimi per questa sezione
from ucimlrepo import fetch_ucirepo                 # scarica dataset UCI
from sklearn.model_selection import train_test_split
import pandas as pd

# 1) Scarico il dataset (id=222 dalle immagini)
bank_marketing = fetch_ucirepo(id=222)

# 2) Estraggo feature e target come DataFrame pandas
X = bank_marketing.data.features    # DataFrame con le feature
y = bank_marketing.data.targets     # DataFrame/Series con il target

# 3) Info di contesto
print(bank_marketing.metadata)      # metadati dataset (descrizione, fonte, ecc.)
print(bank_marketing.variables)     # tabella con info variabili

# 4) (Spesso y è un DataFrame con 1 colonna: lo trasformo in Series)
#    Se y ha già tipo Series, questa riga non crea problemi.
y = y.iloc[:, 0]

# 5) Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape[0]} righe")
print(f"Test: {X_test.shape[0]} righe")


{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [3]:
# ============================================================
# GRUPPO 2 — MODELLI + CROSS VALIDATION + FIT FINALE + PREDIZIONI
# ============================================================

# Import per questa sezione
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# ------------------------------------------------------------
# 1) DEFINIZIONE MODELLI
# ------------------------------------------------------------

# Decision Tree (albero singolo)
tree_model = DecisionTreeClassifier(
    random_state=42
)

# Random Forest (insieme di alberi -> bagging)
rf_model = RandomForestClassifier(
    n_estimators=200,   # nelle immagini c'è 200 (in altre 100: qui uso 200)
    random_state=42,
    n_jobs=-1           # usa tutti i core della CPU
)

# ------------------------------------------------------------
# 2) CROSS VALIDATION (SOLO SUL TRAIN)
#    - Importante: NON tocchiamo X_test / y_test qui.
#    - scoring='roc_auc' perché in immagini usano AUC.
# ------------------------------------------------------------

tree_cv_scores = cross_val_score(
    tree_model,
    X_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

rf_cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

print("Decision Tree CV AUC:", tree_cv_scores.mean())
print("Random Forest CV AUC:", rf_cv_scores.mean())

# ------------------------------------------------------------
# 3) FIT FINALE (SU TUTTO IL TRAIN)
#    - Dopo CV, alleno una volta su tutto il train
# ------------------------------------------------------------

tree_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

# ------------------------------------------------------------
# 4) PREDIZIONI BASE SUL TEST
#    - predict -> classi (0/1)
#    - predict_proba -> probabilità per ogni classe
# ------------------------------------------------------------

tree_pred = tree_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

# Probabilità della classe positiva (colonna 1)
tree_proba = tree_model.predict_proba(X_test)[:, 1]
rf_proba = rf_model.predict_proba(X_test)[:, 1]

# Stampa di controllo (come negli screen)
print("\n--- TEST PERFORMANCE ---")

print("\nDecision Tree")
print("Prime 5 predizioni:", tree_pred[:5])
print("Prime 5 probabilità:", tree_proba[:5].round(3))

print("\nRandom Forest")
print("Prime 5 predizioni:", rf_pred[:5])
print("Prime 5 probabilità:", rf_proba[:5].round(3))

# Controllo forme (in alcune immagini stampano shape)
print("\nShape X_test:", X_test.shape)
print("Shape tree_pred:", tree_pred.shape)


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/root/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/root/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/tree/_classes.py", line 1024, in fit
    super()._fit(
    ~~~~~~~~~~~~^
        X,
        ^^
    ...<2 lines>...
        check_input=check_input,
        ^^^^^^^^^^^^^^^^^^^^^^^^
   

In [ ]:
# ============================================================
# GRUPPO 3 — VALUTAZIONE COMPLETA
# Accuracy • AUC • Classification Report • Confusion Matrix
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) DIZIONARIO MODELLI
#    (Serve per ciclare automaticamente sui modelli)
# ------------------------------------------------------------

models = {
    "Decision Tree": tree_model,
    "Random Forest": rf_model
}

# ------------------------------------------------------------
# 2) CICLO DI VALUTAZIONE
# ------------------------------------------------------------

for name, model in models.items():

    print(f"\n==================== {name} ====================")

    # Predizioni classi
    y_pred = model.predict(X_test)

    # Probabilità classe positiva
    y_proba = model.predict_proba(X_test)[:, 1]

    # -------------------------
    # Accuracy
    # -------------------------
    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", acc)

    # -------------------------
    # AUC ROC
    # -------------------------
    auc_score = roc_auc_score(y_test, y_proba)
    print("AUC:", auc_score)

    # -------------------------
    # Classification Report
    # precision • recall • f1
    # -------------------------
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # ------------------------------------------------------------
    # 3) CONFUSION MATRIX — STAMPA NUMERICA
    # ------------------------------------------------------------

    cm = confusion_matrix(y_test, y_pred)

    print("Confusion Matrix:\n", cm)

    # ------------------------------------------------------------
    # 4) CONFUSION MATRIX — GRAFICO
    # ------------------------------------------------------------

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()

    plt.title(f"{name} - Confusion Matrix")
    plt.show()


In [ ]:
# ============================================================
# GRUPPO 4 — ROC CURVE + PRECISION-RECALL CURVE
# ============================================================

from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Ciclo sui modelli (stesso del gruppo 3)
# ------------------------------------------------------------

for name, model in models.items():

    print(f"\n==================== {name} ====================")

    # Predizioni
    y_pred = model.predict(X_test)

    # Probabilità classe positiva
    y_proba = model.predict_proba(X_test)[:, 1]

    # ========================================================
    # 1) ROC CURVE
    # ========================================================

    # fpr = False Positive Rate
    # tpr = True Positive Rate
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    # Area sotto la curva ROC
    roc_auc = auc(fpr, tpr)

    # ----- Grafico ROC -----

    plt.figure()

    # Curva del modello
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")

    # Linea diagonale (modello casuale)
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} - ROC Curve")
    plt.legend()
    plt.show()

    # ========================================================
    # 2) PRECISION-RECALL CURVE
    # ========================================================

    # precision = TP / (TP + FP)
    # recall    = TP / (TP + FN)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)

    # Area sotto PR curve
    pr_auc = auc(recall, precision)

    # ----- Grafico Precision-Recall -----

    plt.figure()

    plt.plot(recall, precision, label=f"AUC = {pr_auc:.3f}")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{name} - Precision-Recall Curve")
    plt.legend()
    plt.show()


In [ ]:
Probabilità modello
        ↓
ROC Curve
   ├─ fpr
   ├─ tpr
   └─ AUC
        ↓
Grafico ROC

        ↓

Precision-Recall
   ├─ precision
   ├─ recall
   └─ PR AUC
        ↓
Grafico PR


In [ ]:
1️⃣ Gruppo 1 → Import + Dataset + Split
2️⃣ Gruppo 2 → Modelli + Cross Validation + Fit
3️⃣ Gruppo 3 → Accuracy + Report + Confusion Matrix
4️⃣ Gruppo 4 → ROC + Precision-Recall

In [ ]:
# ============================================================
# GRUPPO 5A — VISUALIZZAZIONE DELL'ALBERO (Decision Tree)
# ============================================================

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Nota: qui si usa il modello dt / tree_model già addestrato.
# Se stai seguendo i gruppi precedenti, il tuo modello è "tree_model".

plt.figure(figsize=(20, 8))

plot_tree(
    tree_model,                    # modello Decision Tree già fit-tato
    feature_names=data.feature_names,  # nomi feature dal dataset breast cancer
    class_names=data.target_names,     # nomi classi: ['malignant', 'benign']
    filled=True,                   # colora i nodi in base alla classe prevalente
    rounded=True,                  # arrotonda gli angoli dei box
    fontsize=9
)

plt.title("Decision Tree (max_depth=4)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GRUPPO 5B — FEATURE IMPORTANCE (TOP 10)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Importanze dal Decision Tree (funziona anche con RandomForest)
importances = tree_model.feature_importances_

# Indici delle 10 feature più importanti (ordinate in modo decrescente)
indices = np.argsort(importances)[::-1][:10]  # top 10

plt.figure(figsize=(10, 5))

# Nota: nelle slide c'era color='steelblue'
# Se vuoi esattamente uguale, puoi rimetterlo.
plt.bar(range(10), importances[indices])

plt.xticks(
    range(10),
    data.feature_names[indices],
    rotation=45,
    ha="right"
)

plt.title("Top 10 Feature Importance")
plt.ylabel("Gini Importance")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GRUPPO 5C — ENSEMBLE: BAGGING vs BOOSTING (con cross_val_score)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ------------------------------------------------------------
# 1) CARICAMENTO DATI (come nelle slide)
# ------------------------------------------------------------

data = load_breast_cancer()                 # esempio di dataset
X, y = data.data, data.target              # X = feature, y = target (0/1)

# Nelle slide qui spesso fanno anche train_test_split (anche se poi CV usa X,y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

# ------------------------------------------------------------
# 2) DEFINIZIONE MODELLI
# ------------------------------------------------------------

# Bagging (Random Forest)
bagging_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Boosting (Gradient Boosting)
boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

# Modello base (Decision Tree singolo)
tree_model_base = DecisionTreeClassifier(
    random_state=42
)

# ------------------------------------------------------------
# 3) VALUTAZIONE (Cross Validation su TUTTO X,y nelle slide)
# ------------------------------------------------------------

bagging_scores = cross_val_score(bagging_model, X, y, cv=5)
boosting_scores = cross_val_score(boosting_model, X, y, cv=5)
tree_scores = cross_val_score(tree_model_base, X, y, cv=5)

# ------------------------------------------------------------
# 4) STAMPA RISULTATI
# ------------------------------------------------------------

print(f"Accuratezza del modello Decision Tree: {tree_scores.mean():.4f}")
print(f"Accuratezza media Bagging (Random Forest): {bagging_scores.mean():.4f}")
print(f"Accuratezza media Boosting (Gradient Boosting): {boosting_scores.mean():.4f}")

# ------------------------------------------------------------
# 5) PICCOLO ALBERO SOLO PER PLOT (come nelle slide)
# ------------------------------------------------------------

small_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
small_tree.fit(X, y)

plt.figure(figsize=(20, 10))
plot_tree(
    small_tree,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=12
)
plt.title("Visualizzazione di un Decision Tree (Profondità=3)")
plt.show()


In [ ]:
# ============================================================
# GRUPPO 5D — UCI REPO: BANK MARKETING
# ============================================================

from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)
